# Bài 4.3: Thực hành Tinh chỉnh LoRA / QLoRA cho Chatbot Nội bộ Doanh nghiệp (Company QA)

> **Mục tiêu bài học & Ghi chú học tập (Study Notes):**
> Notebook này được xây dựng dựa trên bộ dữ liệu thực hành **Company QA (`company_docs`, `train.jsonl`, `test.jsonl`)** về quy định nội bộ của *Công ty TNHH Phần mềm Sao Việt*. Qua bài này, bạn sẽ nắm vững từ lý thuyết đến thực chiến:
> 
> 1. **Bài toán thực tế (Grounded QA / RAG Reader Fine-tuning):** Tại sao lại fine-tune LLM cho chatbot nội bộ? Không phải để nhồi kiến thức tĩnh vào trọng số (dễ lỗi thời), mà để **dạy mô hình hành vi chuẩn**:
>    - Chỉ trả lời dựa trên `TÀI LIỆU` được cung cấp trong prompt.
>    - **Luôn trích dẫn nguồn** dạng `[tên_file.md]`.
>    - **Chống ảo giác (Anti-Hallucination):** Biết từ chối trả lời chuẩn câu `"Tài liệu không đề cập thông tin này."` khi ngữ cảnh không chứa đáp án.
>    - **Phân biệt hiệu lực thời gian (Temporal Conflict Resolution):** Khi ngữ cảnh có cả quy định cũ (`*_2023.md`, hết hiệu lực 31/12/2024) và quy định mới (hiệu lực từ 01/01/2025), biết ưu tiên quy định mới và giải thích rõ.
> 2. **Bản chất toán học của LoRA & QLoRA:**
>    - Cách phân rã ma trận cập nhật trọng số $\Delta W = B \cdot A$ với hạng thấp $r \ll \min(d, k)$.
>    - Ý nghĩa thực tế của từng siêu tham số: `r`, `lora_alpha`, `lora_dropout`, `target_modules="all-linear"`.
>    - Sự khác biệt giữa **LoRA (FP16/BF16)** và **QLoRA (4-bit NF4 + Double Quantization + Paged Optimizer)**.
> 3. **Quy trình chuẩn với `transformers`, `peft`, `bitsandbytes`, `trl` (`SFTTrainer`):**
>    - Khám phá dữ liệu (EDA) trên tập `train.jsonl` (496 mẫu) và `test.jsonl` (76 mẫu).
>    - Hiểu cách `apply_chat_template` biến đổi danh sách `messages` (System / User / Assistant) thành chuỗi token chuẩn ChatML.
>    - Kỹ thuật tiết kiệm VRAM: `gradient_checkpointing`, `gradient_accumulation_steps`, `paged_adamw_8bit`.
>    - Đánh giá định tính & định lượng trên `test.jsonl`: So sánh trực tiếp **Model gốc** (`disable_adapter()`) và **Model sau Fine-tune**.

---
## Phần 1: Khám phá Kho Tài liệu Nội bộ (`company_docs`) & Tập dữ liệu (`train.jsonl`, `test.jsonl`)

### 📝 Ghi chú về Thiết kế Dữ liệu (Dataset Design Notes)
Trong hệ thống RAG (Retrieval-Augmented Generation), khi người dùng đặt câu hỏi, bộ truy xuất (Retriever) sẽ lấy ra 1–3 đoạn tài liệu liên quan nhất từ kho `company_docs/` và ghép vào prompt. Tuy nhiên, các model nhỏ (như `Qwen2.5-1.5B-Instruct`) khi chưa fine-tune thường gặp 3 lỗi lớn:
1. **Quên trích dẫn nguồn** hoặc trích dẫn sai định dạng `[tên_file.md]`.
2. **Bịa câu trả lời (Hallucination)** hoặc suy diễn ngoài tài liệu khi tài liệu được truy xuất không chứa thông tin.
3. **Nhầm lẫn văn bản cũ/mới** khi Retriever lấy lên cả file quy định năm 2023 (`*_2023.md`) lẫn file quy định 2025.

Hãy cùng chạy code bên dưới để xem kho tài liệu `company_docs` và cấu trúc của `train.jsonl` / `test.jsonl` được thiết kế như thế nào để giải quyết triệt để 3 lỗi này!

In [2]:
import json
import re
from collections import Counter
from pathlib import Path

# Tự động tìm đường dẫn thư mục dữ liệu (hỗ trợ cả chạy Local trong repo, thư mục Downloads, và trên Kaggle)
CANDIDATE_DIRS = [
    Path("../datasets/lora_company_qa"),
    Path("datasets/lora_company_qa"),
    Path(r"c:/Users/hoangviet/Downloads/lora"),
    Path("/kaggle/input/company-qa"),
]

DATA_DIR = next((p for p in CANDIDATE_DIRS if p.exists()), None)
print(f"📁 Đang sử dụng thư mục dữ liệu: {DATA_DIR.resolve() if DATA_DIR else 'Không tìm thấy'}")

docs_dir = DATA_DIR / "company_docs"
if docs_dir.exists():
    md_files = sorted(docs_dir.glob("*.md"))
    print(f"\n📚 Kho tài liệu company_docs gồm {len(md_files)} file markdown:")
    for f in md_files:
        content = f.read_text(encoding="utf-8").strip()
        first_line = content.splitlines()[0] if content else ""
        tag = "⚠️ [QUY ĐỊNH CŨ - Hết hiệu lực 31/12/2024]" if "2023" in f.name else "✅ [HIỆN HÀNH]"
        print(f"  - {f.name:<22} ({len(content):>3} ký tự) | {first_line:<38} {tag}")

📁 Đang sử dụng thư mục dữ liệu: D:\GithubRepo\LearnNLP\datasets\lora_company_qa

📚 Kho tài liệu company_docs gồm 12 file markdown:
  - bao_mat.md             (312 ký tự) | # Quy định bảo mật thông tin           ✅ [HIỆN HÀNH]
  - co_cau_to_chuc.md      (563 ký tự) | # Cơ cấu tổ chức                       ✅ [HIỆN HÀNH]
  - cong_tac_phi.md        (552 ký tự) | # Quy trình công tác phí               ✅ [HIỆN HÀNH]
  - cong_tac_phi_2023.md   ( 93 ký tự) | # Quy định cũ                          ⚠️ [QUY ĐỊNH CŨ - Hết hiệu lực 31/12/2024]
  - it_faq.md              (603 ký tự) | # Câu hỏi thường gặp về IT             ✅ [HIỆN HÀNH]
  - luong_thuong.md        (516 ký tự) | # Lương, làm thêm giờ và nhân viên mới ✅ [HIỆN HÀNH]
  - mua_sam.md             (380 ký tự) | # Quy trình mua sắm                    ✅ [HIỆN HÀNH]
  - nghi_phep.md           (701 ký tự) | # Quy định nghỉ phép                   ✅ [HIỆN HÀNH]
  - nghi_phep_2023.md      ( 82 ký tự) | # Quy định cũ                          ⚠️ [QUY 

In [3]:
# Đọc và phân tích thống kê tập train.jsonl và test.jsonl
train_path = DATA_DIR / "train.jsonl"
test_path = DATA_DIR / "test.jsonl"

train_samples = [json.loads(line) for line in open(train_path, encoding="utf-8")]
test_samples = [json.loads(line) for line in open(test_path, encoding="utf-8")]

def analyze_split(name, samples):
    total = len(samples)
    # 1. Mẫu từ chối trả lời khi không có thông tin
    no_info = [s for s in samples if "Tài liệu không đề cập" in s["messages"][2]["content"]]
    # 2. Mẫu có tài liệu cũ (*_2023.md) trong ngữ cảnh để dạy mô hình phân biệt hiệu lực
    temporal = [s for s in samples if "_2023.md" in s["messages"][1]["content"]]
    # 3. Mẫu hỏi gộp nhiều ý hoặc tổng hợp từ nhiều file
    citations_per_sample = [
        re.findall(r"\[([a-z0-9_]+\.md)\]", s["messages"][2]["content"])
        for s in samples
    ]
    multi_cite = sum(1 for c in citations_per_sample if len(set(c)) >= 2)
    all_citations = [c for sub in citations_per_sample for c in sub]

    print(f"=== 📊 Thống kê tập {name.upper()} ({total} mẫu) ===")
    print(f"  • Mẫu có câu trả lời trực tiếp : {total - len(no_info):>3} mẫu ({(total - len(no_info))/total*100:.1f}%)")
    print(f"  • Mẫu 'Không đề cập' (Negative): {len(no_info):>3} mẫu ({len(no_info)/total*100:.1f}%) -> Dạy model KHÔNG bịa đặt")
    print(f"  • Mẫu bẫy quy định cũ (2023)   : {len(temporal):>3} mẫu ({len(temporal)/total*100:.1f}%) -> Dạy model ưu tiên bản 2025")
    print(f"  • Mẫu trích dẫn >= 2 tài liệu  : {multi_cite:>3} mẫu")
    print(f"  • Top 5 nguồn được trích dẫn   : {Counter(all_citations).most_common(5)}\n")

analyze_split("train", train_samples)
analyze_split("test", test_samples)

=== 📊 Thống kê tập TRAIN (496 mẫu) ===
  • Mẫu có câu trả lời trực tiếp : 427 mẫu (86.1%)
  • Mẫu 'Không đề cập' (Negative):  69 mẫu (13.9%) -> Dạy model KHÔNG bịa đặt
  • Mẫu bẫy quy định cũ (2023)   :  23 mẫu (4.6%) -> Dạy model ưu tiên bản 2025
  • Mẫu trích dẫn >= 2 tài liệu  :  23 mẫu
  • Top 5 nguồn được trích dẫn   : [('nghi_phep.md', 69), ('luong_thuong.md', 53), ('it_faq.md', 53), ('phuc_loi.md', 53), ('cong_tac_phi.md', 52)]

=== 📊 Thống kê tập TEST (76 mẫu) ===
  • Mẫu có câu trả lời trực tiếp :  66 mẫu (86.8%)
  • Mẫu 'Không đề cập' (Negative):  10 mẫu (13.2%) -> Dạy model KHÔNG bịa đặt
  • Mẫu bẫy quy định cũ (2023)   :   3 mẫu (3.9%) -> Dạy model ưu tiên bản 2025
  • Mẫu trích dẫn >= 2 tài liệu  :   3 mẫu
  • Top 5 nguồn được trích dẫn   : [('nghi_phep.md', 11), ('luong_thuong.md', 8), ('cong_tac_phi.md', 8), ('it_faq.md', 8), ('phuc_loi.md', 8)]



### 🔍 Quan sát 3 dạng mẫu điển hình trong `train.jsonl`
Hãy in ra 3 mẫu thực tế từ tập dữ liệu để thấy rõ cấu trúc `messages` (`system` -> `user` -> `assistant`):

In [5]:
# Tìm 1 mẫu chuẩn, 1 mẫu không có thông tin, và 1 mẫu xung đột quy định 2023 vs 2025
ex_normal = train_samples[0]
ex_no_info = next(s for s in train_samples if "Tài liệu không đề cập" in s["messages"][2]["content"])
ex_temporal = next(s for s in test_samples if "_2023.md" in s["messages"][1]["content"])

for title, sample in [
    ("1️⃣ MẪU TRẢ LỜI & TRÍCH DẪN CHUẨN", ex_normal),
    ("2️⃣ MẪU TỪ CHỐI KHI THIẾU THÔNG TIN (CHỐNG HALLUCINATION)", ex_no_info),
    ("3️⃣ MẪU XỬ LÝ QUY ĐỊNH CŨ (2023) VS MỚI (2025)", ex_temporal),
]:
    print("=" * 85)
    print(title)
    print("=" * 85)
    for msg in sample["messages"]:
        role = msg["role"].upper()
        print(f"[{role}]:\n{msg['content']}\n")

1️⃣ MẪU TRẢ LỜI & TRÍCH DẪN CHUẨN
[SYSTEM]:
Bạn là trợ lý tra cứu thông tin nội bộ. Chỉ trả lời dựa trên phần TÀI LIỆU được cung cấp. Luôn ghi nguồn dạng [tên_file]. Nếu tài liệu không đề cập, hãy nói: 'Tài liệu không đề cập thông tin này.'

[USER]:
TÀI LIỆU:
[co_cau_to_chuc.md] Công ty TNHH Phần mềm Sao Việt do ông Nguyễn Minh Khoa làm Giám đốc điều hành (CEO) và bà Trần Thu Hà làm Giám đốc công nghệ (CTO). Trưởng phòng Nhân sự là ông Lê Quốc Bảo, Trưởng phòng Kế toán là bà Phạm Ngọc Lan, Trưởng phòng IT là ông Võ Đức Anh.

[phuc_loi.md] Thưởng tháng 13 bằng 1 tháng lương cơ bản, trả trước Tết. Thưởng hiệu suất xét hai lần mỗi năm vào tháng 6 và tháng 12. Ngân sách đào tạo là 10.000.000 đồng/người/năm, hoàn tiền sau khi nộp chứng chỉ hoặc hóa đơn khóa học.

[nghi_phep.md] Phép năm chưa dùng được chuyển tối đa 5 ngày sang năm sau và phải dùng hết trước ngày 31/3. Đơn xin nghỉ phép gửi qua HR Portal trước ít nhất 3 ngày làm việc. Nghỉ từ 3 ngày liên tục trở lên phải báo trước 7 ngày.

C

---
## Phần 2: Lý thuyết cốt lõi — LoRA & QLoRA hoạt động như thế nào?

### 1. Tại sao không Full Fine-tuning mà dùng LoRA (Low-Rank Adaptation)?
- Khi **Full Fine-tuning** một mô hình như `Qwen2.5-7B` (7 tỷ tham số), ta phải lưu:
  - Trọng số mô hình (FP16): ~14 GB
  - Gradient (FP16): ~14 GB
  - Trạng thái Optimizer AdamW (FP32 momentum + variance): ~56 GB
  - Activation trung gian: ~10–20 GB
  $\Rightarrow$ Tổng cộng tốn **> 80–100 GB VRAM** (cần A100/H100 đắt đỏ).
- **Ý tưởng của LoRA (Hu et al., 2021):**
  Đóng băng (freeze) toàn bộ ma trận trọng số gốc $W_0 \in \mathbb{R}^{d \times k}$. Thay vì cập nhật trực tiếp $\Delta W \in \mathbb{R}^{d \times k}$, ta xấp xỉ $\Delta W$ bằng tích của hai ma trận hạng thấp (low-rank matrices):
  $$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} B \cdot A$$
  trong đó:
  - $A \in \mathbb{R}^{r \times k}$ (khởi tạo phân phối chuẩn Gaussian $\mathcal{N}(0, \sigma^2)$)
  - $B \in \mathbb{R}^{d \times r}$ (khởi tạo bằng $0$ để tại bước $t=0$, $\Delta W = 0$, mô hình giữ nguyên 100% hành vi gốc)
  - $r \ll \min(d, k)$ là **hạng (rank)**, thường chọn $r \in \{8, 16, 32, 64\}$.
  - $\alpha$ (`lora_alpha`) là hệ số khuếch đại (scaling factor), thường chọn $\alpha = 2r$. Tỉ số $\frac{\alpha}{r}$ giúp khi bạn thay đổi $r$ (ví dụ từ 16 lên 32), bạn không cần tinh chỉnh lại `learning_rate` quá nhiều.

### 2. So sánh nhanh LoRA vs QLoRA (Dettmers et al., 2023)

| Tiêu chí | LoRA (`USE_4BIT = False`) | QLoRA (`USE_4BIT = True`) |
| :--- | :--- | :--- |
| **Định dạng Base Model ($W_0$)** | FP16 hoặc BF16 (16-bit / tham số) | **4-bit NormalFloat (`nf4`)** (~4.5 bit / tham số) |
| **Định dạng Adapter ($A, B$)** | FP16 / BF16 | FP16 / BF16 (vẫn huấn luyện ở độ chính xác cao!) |
| **Quá trình tính toán (Forward/Backward)** | Nhân ma trận trực tiếp ở FP16 | **Dequantize** $W_0$ từ 4-bit $\rightarrow$ FP16 trên on-the-fly để tính toán |
| **Double Quantization** | Không | Lượng tử hóa cả các hằng số lượng tử hóa (tiết kiệm thêm ~0.37 bit/tham số) |
| **Optimizer khuyên dùng** | `adamw_torch` | `paged_adamw_8bit` (tự động đẩy state sang RAM CPU khi VRAM đầy) |
| **Tốc độ train** | **Nhanh hơn** (~20–30%) vì không tốn bước dequantize | Chậm hơn một chút do chi phí giải nén 4-bit $\rightarrow$ 16-bit |
| **Khi nào nên chọn?** | Model nhỏ (`1.5B`, `3B`) vừa khít GPU T4 16GB | Model lớn (`7B`, `8B`, `14B`) hoặc khi VRAM hạn chế (< 12GB) |

---
## Phần 3: Thiết lập Môi trường & Cấu hình Thử nghiệm

> **💡 Lưu ý thực hành khi chạy trên Kaggle hoặc Google Colab (GPU T4 16GB):**
> 1. **Kaggle:** Vào bảng bên phải $\rightarrow$ *Session options* $\rightarrow$ Chọn **Accelerator: GPU T4 x2** (hoặc P100) và bật **Internet: On**.
> 2. **Tại sao đặt `os.environ["CUDA_VISIBLE_DEVICES"] = "0"`?**
>    Kaggle cung cấp 2 GPU T4 (mỗi chiếc 16GB). Nếu không giới hạn về 1 GPU (`"0"`), `device_map="auto"` hoặc `Trainer` (DataParallel) có thể chia model/batch qua 2 GPU và gây lỗi xung đột tensor khác thiết bị với `bitsandbytes` + `gradient_checkpointing`. Chỉ dùng 1 GPU T4 (16GB VRAM) là quá đủ và ổn định nhất cho cả `1.5B` (LoRA) lẫn `7B` (QLoRA).
> 3. **Tại sao dùng `torch.float16` thay vì `bfloat16` trên GPU T4?**
>    GPU NVIDIA Tesla T4 thuộc kiến trúc **Turing (Compute Capability 7.5)**, chỉ hỗ trợ phần cứng tăng tốc cho `fp16`, chưa hỗ trợ `bf16` gốc (chỉ từ Ampere A100/RTX 30xx trở lên mới có `bf16` nhanh).

In [ ]:
# Cài đặt các thư viện cần thiết (Bỏ comment dòng dưới nếu chạy trên Kaggle / Colab)
# !pip install -q -U transformers peft trl bitsandbytes datasets accelerate

In [ ]:
import os
# QUAN TRỌNG: Phải đặt CUDA_VISIBLE_DEVICES trước khi import torch để chỉ định đúng 1 GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import torch

# ================= CẤU HÌNH THỬ NGHIỆM (CHỈNH TẠI ĐÂY) =================
MODEL    = "Qwen/Qwen2.5-1.5B-Instruct"   # Có thể đổi sang: "Qwen/Qwen2.5-7B-Instruct"
USE_4BIT = False                           # False = LoRA (fp16), True = QLoRA (4-bit NF4, bắt buộc với 7B trên T4)
OUT_DIR  = "./adapter_out"                 # Trên Kaggle có thể đặt: "/kaggle/working/adapter_out"
MAX_LEN  = 1024                            # Độ dài chuỗi token tối đa (context + câu hỏi + câu trả lời)
# =======================================================================

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"✅ GPU sẵn sàng: {gpu_name} | Tổng VRAM: {vram_gb} GB")
else:
    print("⚠️ Không phát hiện GPU CUDA. Bạn vẫn có thể chạy các cell khám phá dữ liệu & kiểm tra template bên dưới!")

---
## Phần 4: Chuẩn bị Dataset cho `SFTTrainer`

### 📝 Ghi chú học tập:
- Trong notebook gốc trên Kaggle, nếu chưa upload file `train.jsonl`, code sẽ dùng 5 mẫu giả nhân bản 20 lần (`* 20`) chỉ để test luồng chạy.
- Ở đây, chúng ta đã có sẵn **bộ dữ liệu thật** gồm **`train.jsonl` (496 mẫu)** và **`test.jsonl` (76 mẫu)** độc lập!
- Việc tách riêng `train.jsonl` và `test.jsonl` (thay vì nhân bản mẫu) giúp chỉ số `eval_loss` sau mỗi epoch phản ánh trung thực khả năng tổng quát hóa của mô hình và giúp phát hiện sớm hiện tượng **Overfitting**.
- Thư viện `trl` (`SFTTrainer`) hỗ trợ trực tiếp cột `messages` (danh sách các dict `{"role": ..., "content": ...}`) và sẽ **tự động gọi `tokenizer.apply_chat_template()`** của dòng model tương ứng.

In [ ]:
from datasets import Dataset, DatasetDict

SYSTEM = (
    "Bạn là trợ lý tra cứu thông tin nội bộ. Chỉ trả lời dựa trên phần TÀI LIỆU được cung cấp. "
    "Luôn ghi nguồn dạng [tên_file]. Nếu tài liệu không đề cập, hãy nói: 'Tài liệu không đề cập thông tin này.'"
)

def ex(ctx, q, a):
    """Hàm tiện ích tạo nhanh 1 mẫu dữ liệu chuẩn định dạng messages."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"TÀI LIỆU:\n{ctx}\n\nCÂU HỎI: {q}"},
            {"role": "assistant", "content": a},
        ]
    }

# Tự động nạp train.jsonl & test.jsonl từ thư mục local hoặc Kaggle Input
if DATA_DIR and (DATA_DIR / "train.jsonl").exists():
    train_data = [json.loads(l) for l in open(DATA_DIR / "train.jsonl", encoding="utf-8")]
    if (DATA_DIR / "test.jsonl").exists():
        test_data = [json.loads(l) for l in open(DATA_DIR / "test.jsonl", encoding="utf-8")]
        ds = DatasetDict({
            "train": Dataset.from_list(train_data),
            "test": Dataset.from_list(test_data),
        })
        print(f"✅ Đã nạp thành công train.jsonl ({len(train_data)} mẫu) và test.jsonl ({len(test_data)} mẫu)!")
    else:
        ds = Dataset.from_list(train_data).train_test_split(test_size=0.1, seed=42)
        print(f"✅ Đã nạp train.jsonl ({len(train_data)} mẫu) và chia 90/10 train/test!")
else:
    # Fallback 5 mẫu mẫu nếu chạy ở môi trường chưa có file jsonl
    print("⚠️ Không tìm thấy train.jsonl, dùng dữ liệu mẫu fallback để kiểm tra pipeline:")
    fallback_samples = [
        ex("[nghi_phep.md] Nhân viên chính thức có 12 ngày phép năm. Phép chưa dùng được chuyển tối đa 5 ngày sang năm sau.",
           "Tôi được bao nhiêu ngày phép một năm?", "Nhân viên chính thức có 12 ngày phép năm. [nghi_phep.md]"),
        ex("[nghi_phep.md] Nhân viên chính thức có 12 ngày phép năm. Phép chưa dùng được chuyển tối đa 5 ngày sang năm sau.",
           "Công ty có hỗ trợ tiền gửi xe không?", "Tài liệu không đề cập thông tin này."),
    ] * 20
    ds = Dataset.from_list(fallback_samples).train_test_split(test_size=0.1, seed=42)

print(ds)

---
## Phần 5: Load Tokenizer & Model (Hỗ trợ cả LoRA FP16 và QLoRA 4-bit)

### 📝 Ghi chú kỹ thuật từng dòng code:
1. **`tok.pad_token = tok.eos_token`**:
   Nhiều mô hình Causal LM (đặc biệt họ Llama/Mistral) không định nghĩa sẵn `pad_token` riêng. Khi gom nhiều câu có độ dài khác nhau vào cùng một batch, Tokenizer bắt buộc phải có `pad_token` để đệm (padding) các câu ngắn cho bằng câu dài nhất.
2. **Cấu hình `BitsAndBytesConfig` (khi `USE_4BIT = True`)**:
   - `load_in_4bit=True`: Nén trọng số base model xuống 4-bit khi nạp vào VRAM.
   - `bnb_4bit_quant_type="nf4"`: Dùng kiểu dữ liệu **4-bit NormalFloat (NF4)** — tối ưu về mặt lý thuyết thông tin cho các trọng số mạng nơ-ron tuân theo phân phối chuẩn $\mathcal{N}(0, \sigma^2)$.
   - `bnb_4bit_use_double_quant=True`: Lượng tử hóa lần thứ 2 đối với các hằng số scale (quantization constants) của từng block 64 trọng số, giúp tiết kiệm thêm ~0.37 bit/tham số (~350MB VRAM cho model 7B).
   - `bnb_4bit_compute_dtype=torch.float16`: Dù lưu trữ ở 4-bit, khi thực hiện phép nhân ma trận trong lượt Forward/Backward, trọng số sẽ được giải nén tạm thời ra `float16` để đảm bảo độ chính xác gradient.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Load Tokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Minh họa cách Tokenizer biến đổi 1 mẫu messages thành chuỗi ChatML thực tế đưa vào model
sample_text = tok.apply_chat_template(ds["train"][0]["messages"], tokenize=False)
sample_tokens = tok.apply_chat_template(ds["train"][0]["messages"], tokenize=True)
print(f"🔍 Minh họa Chat Template của {MODEL} (Độ dài: {len(sample_tokens)} tokens):")
print("-" * 80)
print(sample_text)
print("-" * 80)

In [ ]:
# 2. Cấu hình Lượng tử hóa 4-bit (nếu USE_4BIT = True)
bnb = None
if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,   # Dùng float16 để tương thích tốt nhất với GPU T4
    )

# 3. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map="auto",
)

if torch.cuda.is_available():
    print("📊 VRAM sau khi load Base Model:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

---
## Phần 6: Cấu hình LoRA (`LoraConfig`) — Giải thích chi tiết từng tham số

```python
peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)
```

### 📝 Ghi chú chuyên sâu về các tham số trong `LoraConfig`:
1. **`r=16` (Rank - Hạng của ma trận LoRA):**
   - Quyết định kích thước chiều ẩn của 2 ma trận $A \in \mathbb{R}^{r \times k}$ và $B \in \mathbb{R}^{d \times r}$.
   - `r` càng lớn $\rightarrow$ số tham số huấn luyện (trainable parameters) càng nhiều, khả năng ghi nhớ chi tiết càng cao nhưng **dễ bị Overfit** nếu tập dữ liệu nhỏ (như ~500 mẫu của chúng ta).
   - Kinh nghiệm thực tế: Với bài toán dạy định dạng trả lời & trích dẫn nguồn (behavior alignment), `r=8` hoặc `r=16` là điểm cân bằng vàng.
2. **`lora_alpha=32` (Scaling factor):**
   - Đầu ra của nhánh LoRA được nhân với hệ số $\frac{\text{lora\_alpha}}{r} = \frac{32}{16} = 2.0$ trước khi cộng vào đầu ra của lớp Linear gốc.
   - Quy tắc ngón tay cái (Rule of thumb): Đặt `lora_alpha = 2 * r` (hoặc `lora_alpha = r`).
3. **`target_modules="all-linear"`:**
   - Thay vì chỉ gắn LoRA vào 2 lớp `q_proj`, `v_proj` của khối Self-Attention như bài báo LoRA đời đầu, bài báo **QLoRA** chỉ ra rằng gắn adapter vào **tất cả các lớp Linear** (`q_proj`, `k_proj`, `v_proj`, `o_proj` trong Attention VÀ `gate_proj`, `up_proj`, `down_proj` trong mạng MLP/Feed-Forward) cho chất lượng tương đương Full Fine-tuning nhất.
   - Trong `peft` mới, chỉ cần truyền chuỗi `"all-linear"`, thư viện sẽ tự động tìm mọi lớp Linear (trừ lớp đầu ra `lm_head`) để gắn Adapter.
4. **`lora_dropout=0.05` & `bias="none"`:**
   - `lora_dropout=0.05`: Triệt tiêu ngẫu nhiên 5% tín hiệu đi qua nhánh LoRA trong lúc train để chống Overfitting.
   - `bias="none"`: Không huấn luyện các vector hệ số chệch (bias), giữ nguyên hoàn toàn của model gốc để tiết kiệm bộ nhớ.

In [ ]:
from peft import LoraConfig

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

---
## Phần 7: Cấu hình Huấn luyện (`SFTConfig` & `SFTTrainer`)

### 📝 Ghi chú về các kỹ thuật tối ưu VRAM và hội tụ trong `SFTConfig`:
- **Batch Size hiệu dụng (`Effective Batch Size`)**:
  - Nếu đặt `batch_size = 16` trực tiếp, GPU 16GB sẽ báo lỗi `CUDA Out of Memory (OOM)` ngay lập tức.
  - Giải pháp: Đặt `per_device_train_batch_size = 2` kết hợp `gradient_accumulation_steps = 8`. Mô hình sẽ tính gradient trên từng lô nhỏ 2 mẫu, cộng dồn (accumulate) suốt 8 lần rồi mới cập nhật trọng số 1 lần. Kết quả toán học tương đương với việc train ở **Batch Size hiệu dụng = $2 \times 8 = 16$** mà chỉ tốn VRAM của 2 mẫu!
- **`gradient_checkpointing=True`:**
  - Không lưu toàn bộ các ma trận kích hoạt trung gian (intermediate activations) của cả 28–32 tầng Transformer trong lượt Forward, mà chỉ lưu một số điểm chốt (checkpoints) và tính lại phần thiếu trong lượt Backward. Đổi ~20% thời gian tính toán để **giảm tới 60–70% VRAM activation**.
- **`optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch"`:**
  - Khi bật QLoRA (`USE_4BIT=True`), bộ tối ưu `paged_adamw_8bit` nén trạng thái momentum/variance của Adam xuống 8-bit và dùng cơ chế Unified Memory của CUDA để tự động "tràn" sang RAM hệ thống nếu có bước nào chuỗi dài làm VRAM tăng đột biến.
- **`learning_rate=2e-4` & `lr_scheduler_type="cosine"`:**
  - Với LoRA, tốc độ học `2e-4` ($0.0002$) cao gấp ~10 lần so với Full Fine-tuning (`1e-5` - `2e-5`) vì các ma trận $A, B$ mới khởi tạo cần bước nhảy lớn hơn. Lịch giảm tốc dạng hình sin (`cosine`) kết hợp `warmup_ratio=0.05` (tăng dần LR trong 5% số bước đầu) giúp mô hình hội tụ êm, không làm hỏng kiến thức ngôn ngữ sẵn có.

In [ ]:
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,     # Batch size hiệu dụng = 2 * 8 = 16
    learning_rate=2e-4,                # Nếu thấy eval_loss dao động mạnh, có thể hạ xuống 1e-4
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,       # Bắt buộc bật để tiết kiệm VRAM
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    logging_steps=5,
    eval_strategy="epoch",             # Đánh giá trên tập test sau mỗi epoch
    save_strategy="no",                # Tiết kiệm ổ cứng trong lúc train, lưu thủ công ở cuối
    max_length=MAX_LEN,                # Lưu ý: các bản TRL cũ (<0.12) dùng tên tham số max_seq_length
    report_to="none",                  # Tắt WandB/TensorBoard mặc định để không bị hỏi API key
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=peft_cfg,
    processing_class=tok,
)

# In ra số lượng tham số được huấn luyện so với tổng số tham số của mô hình
trainer.model.print_trainable_parameters()

> **💡 Giải nghĩa kết quả `print_trainable_parameters()`:**
> Với `Qwen2.5-1.5B-Instruct` và `r=16`, bạn sẽ thấy chỉ có khoảng **~18.4 triệu tham số** được huấn luyện trên tổng số **~1.56 tỷ tham số** (tức chỉ khoảng **~1.18%**!).
> Nhờ vậy, khi lưu kết quả ở cell tiếp theo, file trọng số Adapter (`adapter_model.safetensors`) chỉ nặng **vài chục MB** thay vì 3 GB như model gốc.

In [ ]:
# Bắt đầu quá trình Fine-tuning
train_result = trainer.train()

if torch.cuda.is_available():
    print("📊 VRAM đỉnh (Peak VRAM) trong lúc train:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")

# Lưu LoRA Adapter và Tokenizer ra thư mục OUT_DIR
trainer.model.save_pretrained(OUT_DIR)   # Chỉ lưu trọng số của Adapter LoRA (vài chục MB)
tok.save_pretrained(OUT_DIR)
print(f"✅ Đã lưu LoRA Adapter và Tokenizer tại: {OUT_DIR}")

---
## Phần 8: Kiểm thử & So sánh Trực tiếp: Model Gốc vs Model Đã Fine-tune

### 📝 Mẹo hay với `peft`: `with m.disable_adapter():`
- Thông thường, để so sánh model trước và sau khi fine-tune, nhiều người phải nạp 2 model riêng biệt vào VRAM (tốn gấp đôi bộ nhớ).
- Với kiến trúc LoRA ($W = W_0 + \Delta W$), trọng số gốc $W_0$ vẫn nằm nguyên vẹn bên dưới! Do đó, thư viện `peft` cung cấp context manager **`with m.disable_adapter():`** giúp tạm thời tắt nhánh $\Delta W = B \cdot A$ để chạy đúng 100% như **Base Model gốc** mà không tốn thêm một byte VRAM nào.

### 🎯 Tiêu chí quan sát khi chạy so sánh:
1. **Định dạng trích dẫn:** Model đã fine-tune có luôn gắn đúng `[tên_file.md]` ở cuối câu không?
2. **Khả năng từ chối:** Khi hỏi câu không có trong tài liệu (ví dụ hỏi *nghỉ lễ* nhưng tài liệu chỉ nói về *IT FAQ*), model gốc thường tự lấy kiến thức ngoài đời trả lời bừa, còn model đã fine-tune sẽ trả lời đúng chuẩn: `"Tài liệu không đề cập thông tin này."`
3. **Phân biệt quy định cũ / mới:** Khi có cả `[cong_tac_phi_2023.md]` và `[cong_tac_phi.md]` (2025), model đã fine-tune có chọn đúng mức `150.000 đồng` và nhắc bản 2023 đã hết hiệu lực không?

In [ ]:
def ask(ctx, q, use_adapter=True, max_new_tokens=200):
    """
    Hàm đặt câu hỏi dựa trên tài liệu ctx.
    - use_adapter=True : Dùng model đã fine-tune LoRA
    - use_adapter=False: Tắt LoRA adapter -> Chạy đúng Base Model gốc để đối chiếu
    """
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"TÀI LIỆU:\n{ctx}\n\nCÂU HỎI: {q}"},
    ]
    ids = tok.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    m = trainer.model
    m.eval()
    with torch.no_grad():
        if use_adapter:
            out = m.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
        else:
            with m.disable_adapter():
                out = m.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)

    # Chỉ cắt lấy phần token mới sinh ra của assistant (bỏ phần prompt đầu vào)
    generated_tokens = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(generated_tokens, skip_special_tokens=True)


# --- KỊCH BẢN TEST 1: Câu hỏi có trong tài liệu & Câu hỏi ngoài tài liệu ---
ctx_1 = "[it_faq.md] Quên mật khẩu email: vào portal.congty.vn/reset. Laptop hỏng: gửi ticket cho IT trong 24 giờ."
tests_1 = [
    "Quên mật khẩu email thì làm sao?",
    "Công ty có bao nhiêu ngày nghỉ lễ?",  # Bẫy: Không có trong tài liệu!
]

print("================ KỊCH BẢN 1: TRÍCH DẪN & CHỐNG BỊA ĐẶT ================")
for q in tests_1:
    print(f"❓ Câu hỏi : {q}")
    print(f"  🔹 GỐC      : {ask(ctx_1, q, use_adapter=False)}")
    print(f"  🔸 FINETUNE : {ask(ctx_1, q, use_adapter=True)}")
    print()

# --- KỊCH BẢN TEST 2: Xung đột quy định cũ (2023) và mới (2025) từ tập test.jsonl ---
ctx_2 = (
    "[cong_tac_phi_2023.md] (Hiệu lực đến 31/12/2024) Phụ cấp ăn là 120.000 đồng/ngày công tác trong nước.\n\n"
    "[cong_tac_phi.md] (Hiệu lực từ 01/01/2025) Phụ cấp ăn là 150.000 đồng/ngày công tác trong nước."
)
q_2 = "Phụ cấp ăn công tác là 120.000 hay 150.000 đồng?"

print("================ KỊCH BẢN 2: PHÂN BIỆT QUY ĐỊNH CŨ (2023) VS MỚI (2025) ================")
print(f"❓ Câu hỏi : {q_2}")
print(f"  🔹 GỐC      : {ask(ctx_2, q_2, use_adapter=False)}")
print(f"  🔸 FINETUNE : {ask(ctx_2, q_2, use_adapter=True)}")

### 📈 Đánh giá Định lượng (Quantitative Evaluation) trên tập `test.jsonl`
Ngoài việc xem bằng mắt vài câu hỏi, trong thực tế bạn nên đo lường tự động trên tập `test.jsonl` theo 2 chỉ số quan trọng nhất của bài toán này:
1. **Tỉ lệ trích dẫn đúng nguồn (`Citation Accuracy`)** trên các câu có đáp án.
2. **Tỉ lệ từ chối chính xác (`Abstention Accuracy`)** trên các câu không có thông tin trong tài liệu.

In [ ]:
def evaluate_on_test_subset(num_samples=20):
    """Đánh giá nhanh model gốc vs model fine-tune trên num_samples mẫu đầu của tập test."""
    subset = list(ds["test"])[:num_samples]
    stats = {
        "base": {"cite_ok": 0, "abstain_ok": 0},
        "lora": {"cite_ok": 0, "abstain_ok": 0},
    }
    total_answerable = 0
    total_unanswerable = 0

    for item in subset:
        user_text = item["messages"][1]["content"]
        gold_answer = item["messages"][2]["content"]

        # Tách phần TÀI LIỆU và CÂU HỎI từ user_text
        parts = user_text.split("\n\nCÂU HỎI: ")
        ctx = parts[0].replace("TÀI LIỆU:\n", "", 1)
        q = parts[1] if len(parts) > 1 else ""

        pred_base = ask(ctx, q, use_adapter=False)
        pred_lora = ask(ctx, q, use_adapter=True)

        is_unanswerable = "Tài liệu không đề cập" in gold_answer
        if is_unanswerable:
            total_unanswerable += 1
            if "không đề cập" in pred_base.lower():
                stats["base"]["abstain_ok"] += 1
            if "không đề cập" in pred_lora.lower():
                stats["lora"]["abstain_ok"] += 1
        else:
            total_answerable += 1
            gold_cites = set(re.findall(r"\[([a-z0-9_]+\.md)\]", gold_answer))
            base_cites = set(re.findall(r"\[([a-z0-9_]+\.md)\]", pred_base))
            lora_cites = set(re.findall(r"\[([a-z0-9_]+\.md)\]", pred_lora))
            if gold_cites and gold_cites.issubset(base_cites):
                stats["base"]["cite_ok"] += 1
            if gold_cites and gold_cites.issubset(lora_cites):
                stats["lora"]["cite_ok"] += 1

    print(f"📊 KẾT QUẢ ĐÁNH GIÁ TRÊN {len(subset)} MẪU TEST:")
    if total_answerable:
        print(f"  • Trích dẫn đúng nguồn [file.md] ({total_answerable} câu):")
        print(f"      - Model Gốc      : {stats['base']['cite_ok']}/{total_answerable} ({stats['base']['cite_ok']/total_answerable*100:.1f}%)")
        print(f"      - Model Finetune : {stats['lora']['cite_ok']}/{total_answerable} ({stats['lora']['cite_ok']/total_answerable*100:.1f}%)")
    if total_unanswerable:
        print(f"  • Từ chối đúng khi thiếu thông tin ({total_unanswerable} câu):")
        print(f"      - Model Gốc      : {stats['base']['abstain_ok']}/{total_unanswerable} ({stats['base']['abstain_ok']/total_unanswerable*100:.1f}%)")
        print(f"      - Model Finetune : {stats['lora']['abstain_ok']}/{total_unanswerable} ({stats['lora']['abstain_ok']/total_unanswerable*100:.1f}%)")

# Bỏ comment dòng dưới để chạy đánh giá sau khi đã train xong:
# evaluate_on_test_subset(num_samples=20)

---
## Phần 9: Đóng gói Adapter, Merge Model & Hướng dẫn Bước tiếp theo

### 1. Lưu và tải Adapter từ Kaggle
Sau khi chạy `trainer.model.save_pretrained(OUT_DIR)`, thư mục `OUT_DIR` chỉ chứa:
- `adapter_config.json` (cấu hình `r`, `lora_alpha`, `target_modules`...)
- `adapter_model.safetensors` (trọng số ma trận $A, B$ — chỉ vài chục MB)
- Các file của Tokenizer (`tokenizer.json`, `tokenizer_config.json`...)

### 2. Khi nào cần Merge Adapter vào Base Model?
- Khi suy luận bằng `vLLM` hoặc `peft`, bạn có thể giữ nguyên Base Model + nạp Adapter động.
- Nếu muốn xuất sang định dạng **GGUF (`llama.cpp` / `Ollama`)** để chạy trên CPU/MacBook hoặc nhúng vào pipeline RAG nội bộ, bạn cần **gộp (merge)** trọng số $\Delta W = \frac{\alpha}{r} B \cdot A$ cộng thẳng vào $W_0$:
  ```python
  from peft import AutoPeftModelForCausalLM
  import torch

  # Lưu ý: Luôn merge ở độ chính xác FP16/BF16 (KHÔNG merge trực tiếp trên model đang load 4-bit)
  peft_model = AutoPeftModelForCausalLM.from_pretrained(
      OUT_DIR,
      torch_dtype=torch.float16,
      device_map="auto",
  )
  merged_model = peft_model.merge_and_unload()
  merged_model.save_pretrained("./merged_qwen_company_qa")
  tok.save_pretrained("./merged_qwen_company_qa")
  ```

### 3. Bài tập tự thực hành nâng cao:
1. **Thử nghiệm Hyperparameter:** Thử thay đổi `r = 8 / 16 / 64` và `num_train_epochs = 1 / 3 / 5`, ghi lại bảng `train_loss` và `eval_loss` sau mỗi epoch để quan sát thời điểm `eval_loss` bắt đầu tăng ngược trở lại (dấu hiệu Overfitting).
2. **So sánh LoRA vs QLoRA trên 7B:** Đổi `MODEL = "Qwen/Qwen2.5-7B-Instruct"` và bật `USE_4BIT = True`, đối chiếu lượng **VRAM sau khi load**, **VRAM đỉnh khi train**, và **chất lượng câu trả lời** so với bản `1.5B`.
3. **Tích hợp vào RAG hoàn chỉnh:** Kết hợp model đã fine-tune này với bộ truy xuất BM25 + Vector Search trên thư mục `company_docs/` (xem thêm Bài 4.2) để tạo thành hệ thống Chatbot Nội bộ hoàn chỉnh!

In [ ]:
# Nếu chạy trên Kaggle: Nén thư mục adapter_out thành file .zip để tải về máy cá nhân dễ dàng
import shutil
from pathlib import Path

if Path(OUT_DIR).exists():
    zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
    size_mb = Path(zip_path).stat().st_size / (1024 * 1024)
    print(f"📦 Đã nén thành công: {zip_path} ({size_mb:.2f} MB)")